Logistic Regression

In [ ]:
import numpy as np
import pandas as pd
import warnings
import json
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    f1_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
# 0.  CONFIG
# ══════════════════════════════════════════════════════════════════
DATA_DIR     = "."
OUTPUT_DIR   = "."
RANDOM_STATE = 42
CV_FOLDS     = 5
SCORING      = "roc_auc"

# ══════════════════════════════════════════════════════════════════
# 1.  LOAD DATA
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 │ Load preprocessed data")
print("=" * 65)

X_train = pd.read_csv('X_train_final.csv', index_col=0)
y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
X_test  = pd.read_csv('X_test_final.csv',  index_col=0)
y_test  = pd.read_csv('y_test_final.csv',  index_col=0)['icu_death_flag']

print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train distribution:\n{y_train.value_counts()}")
print(f"  y_test  distribution:\n{y_test.value_counts()}\n")

pos_rate   = y_train.mean()
imbalanced = pos_rate < 0.2 or pos_rate > 0.8
print(f"  Positive rate (train): {pos_rate:.3f}  "
      f"→ {'Imbalanced, will add class_weight=balanced' if imbalanced else 'Balanced'}\n")

# ══════════════════════════════════════════════════════════════════
# 2.  DEFINE GRID
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 2 │ Define hyperparameter grid")
print("=" * 65)

base_params = {"max_iter": 2000, "random_state": RANDOM_STATE}
if imbalanced:
    base_params["class_weight"] = "balanced"

param_grid = [
    # L1
    {
        "penalty" : ["l1"],
        "C"       : [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
        "solver"  : ["liblinear", "saga"],
    },
    # L2
    {
        "penalty" : ["l2"],
        "C"       : [0.001, 0.005, 0.01, 0.05, 0.1, 0.5, 1.0, 5.0, 10.0],
        "solver"  : ["lbfgs", "liblinear"],
    },
    # ElasticNet
     {   "penalty"  : ["elasticnet"],
        "C"        : [0.01, 0.1, 1.0, 5.0],
        "solver"   : ["saga"],
        "l1_ratio" : [0.1, 0.3, 0.5, 0.7, 0.9],
    },
]

total_combos = sum(np.prod([len(v) for v in g.values()]) for g in param_grid)
print(f"  Total combinations : {total_combos}")
print(f"  CV folds           : {CV_FOLDS}")
print(f"  Scoring metric     : {SCORING}")
print(f"  Total fits         : {total_combos * CV_FOLDS}\n")

# ══════════════════════════════════════════════════════════════════
# 3.  GRID SEARCH
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 3 │ Running GridSearchCV")
print("=" * 65)

cv     = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)
lr_base = LogisticRegression(**base_params)

gs = GridSearchCV(
    estimator  = lr_base,
    param_grid = param_grid,
    cv         = cv,
    scoring    = SCORING,
    n_jobs     = -1,
    verbose    = 1,
    refit      = True,
    return_train_score = True,
)

gs.fit(X_train, y_train)

print(f"\n  ✓ Best {SCORING} (CV) : {gs.best_score_:.4f}")
print(f"  Best params        : {gs.best_params_}\n")

# ══════════════════════════════════════════════════════════════════
# 4.  EVALUATE ON TEST SET
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 4 │ Evaluate best model on Test set")
print("=" * 65)

best_model = gs.best_estimator_
y_pred     = best_model.predict(X_test)
y_prob     = best_model.predict_proba(X_test)[:, 1]

# 只保留核心分类评估指标
metrics = {
    "accuracy" : accuracy_score(y_test, y_pred),
    "roc_auc"  : roc_auc_score(y_test, y_prob),
    "f1"       : f1_score(y_test, y_pred),
    "f1_macro" : f1_score(y_test, y_pred, average="macro"),
}

for k, v in metrics.items():
    print(f"  {k:<20s}: {v:.4f}")

print(f"\n  Classification Report:\n")
print(classification_report(y_test, y_pred))

# ══════════════════════════════════════════════════════════════════
# 5.  FEATURE IMPORTANCE  (non-zero coefficients)
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 5 │ Feature importance (coefficients)")
print("=" * 65)

coef = best_model.coef_[0]
coef_df = pd.DataFrame({
    "feature"  : X_train.columns.tolist(),
    "coef"     : coef,
    "abs_coef" : np.abs(coef),
}).sort_values("abs_coef", ascending=False).reset_index(drop=True)

n_nonzero = (coef_df["coef"] != 0).sum()
print(f"  Total features     : {len(coef_df)}")
print(f"  Non-zero features  : {n_nonzero}  "
      f"({'L1/ElasticNet sparse solution' if n_nonzero < len(coef_df) else 'L2 keeps all'})")
print(f"\n  Top 20 features by |coefficient|:")
print(coef_df.head(20).to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# 6.  PLOTS: ROC Curve + Confusion Matrix
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 6 │ Generating plots")
print("=" * 65)

plt.style.use("dark_background")
fig = plt.figure(figsize=(12, 5))
fig.patch.set_facecolor("#0d0f14")
gs_plot = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

ACCENT = "#00e5a0"
PANEL  = "#14161e"

def panel_style(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors="#c8d0e8", labelsize=8)
    ax.xaxis.label.set_color("#c8d0e8")
    ax.yaxis.label.set_color("#c8d0e8")
    ax.title.set_color("#ffffff")
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#252836")

# ── ROC Curve ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs_plot[0, 0])
RocCurveDisplay.from_predictions(y_test, y_prob, ax=ax1,
    color=ACCENT, lw=2,
    name=f"LR (AUC={metrics['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], "--", color="#5a6080", lw=1)
ax1.legend(fontsize=8, framealpha=0.2)
panel_style(ax1, "ROC Curve (Test)")

# ── Confusion Matrix ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs_plot[0, 1])
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", ax=ax2,
    cmap="YlOrRd", linewidths=0.5,
    annot_kws={"size": 13, "weight": "bold"},
    cbar_kws={"shrink": 0.8})
ax2.set_xlabel("Predicted", fontsize=9)
ax2.set_ylabel("Actual", fontsize=9)
panel_style(ax2, "Confusion Matrix (Test)")

fig.suptitle("Logistic Regression — Classification Report",
             fontsize=13, fontweight="bold", color="#ffffff", y=1.02)

plot_path = f"{OUTPUT_DIR}/lr_gridsearch_report.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  → Saved plot : {plot_path}\n")

# ══════════════════════════════════════════════════════════════════
# 7.  SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 7 │ Save outputs")
print("=" * 65)

# Best params + metrics → JSON
summary = {
    "best_params"       : gs.best_params_,
    "cv_best_score"     : round(gs.best_score_, 6),
    "test_metrics"      : {k: round(v, 6) for k, v in metrics.items()},
    "n_features_total"  : int(len(coef_df)),
    "n_features_nonzero": int(n_nonzero),
}
with open(f"{OUTPUT_DIR}/lr_best_params.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"  → lr_best_params.json")


# Selected features (non-zero coef)
selected_features = coef_df[coef_df["coef"] != 0]["feature"].tolist()


print()
print("=" * 65)
print("  Grid Search complete.")
print(f"  Best model : {gs.best_params_}")
print(f"  Test AUC   : {metrics['roc_auc']:.4f}")
print("=" * 65)

STEP 1 │ Load preprocessed data
  X_train : (45755, 37)
  X_test  : (19610, 37)
  y_train distribution:
icu_death_flag
0    41785
1     3970
Name: count, dtype: int64
  y_test  distribution:
icu_death_flag
0    17909
1     1701
Name: count, dtype: int64

  Positive rate (train): 0.087  → Imbalanced, will add class_weight=balanced

STEP 2 │ Define hyperparameter grid
  Total combinations : 56
  CV folds           : 5
  Scoring metric     : roc_auc
  Total fits         : 280

STEP 3 │ Running GridSearchCV  (this may take a few minutes...)
Fitting 5 folds for each of 56 candidates, totalling 280 fits

  ✓ Best roc_auc (CV) : 0.9046
  Best params        : {'C': 0.1, 'penalty': 'l2', 'solver': 'lbfgs'}

STEP 4 │ Evaluate best model on Test set
  accuracy            : 0.8380
  roc_auc             : 0.9058
  f1                  : 0.4663
  f1_macro            : 0.6854

  Classification Report:

              precision    recall  f1-score   support

           0       0.98      0.84      0.90  

Desicion Tree

In [ ]:
import numpy as np
import pandas as pd
import warnings
import json
from pathlib import Path

from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    f1_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
# 0.  CONFIG
# ══════════════════════════════════════════════════════════════════
DATA_DIR     = "."
OUTPUT_DIR   = "."
RANDOM_STATE = 42
CV_FOLDS     = 5
SCORING      = "roc_auc"

# ══════════════════════════════════════════════════════════════════
# 1.  LOAD DATA
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 │ Load preprocessed data")
print("=" * 65)

X_train = pd.read_csv('X_train_final.csv', index_col=0)
y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
X_test  = pd.read_csv('X_test_final.csv',  index_col=0)
y_test  = pd.read_csv('y_test_final.csv',  index_col=0)['icu_death_flag']

print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train distribution:\n{y_train.value_counts()}")
print(f"  y_test  distribution:\n{y_test.value_counts()}\n")

pos_rate   = y_train.mean()
imbalanced = pos_rate < 0.2 or pos_rate > 0.8
print(f"  Positive rate (train): {pos_rate:.3f}  "
      f"→ {'Imbalanced, will add class_weight=balanced' if imbalanced else 'Balanced'}\n")

feature_names = X_train.columns.tolist()

# ══════════════════════════════════════════════════════════════════
# 2.  DEFINE GRID
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 2 │ Define hyperparameter grid")
print("=" * 65)

param_grid = {
    "criterion"         : ["gini", "entropy"],
    "max_depth"         : [3, 5, 7, 10, 15, None],
    "min_samples_split" : [2, 10, 20, 50],
    "min_samples_leaf"  : [1, 5, 10, 20],
    "max_features"      : [None, "sqrt", "log2"],
    "ccp_alpha"         : [0.0, 0.0001, 0.001, 0.01],
}

total_combos = int(np.prod([len(v) for v in param_grid.values()]))
print(f"  Total combinations : {total_combos}")
print(f"  CV folds           : {CV_FOLDS}")
print(f"  Scoring metric     : {SCORING}")
print(f"  Total fits         : {total_combos * CV_FOLDS}\n")

for k, v in param_grid.items():
    print(f"  {k:<25s}: {v}")

# ══════════════════════════════════════════════════════════════════
# 3.  GRID SEARCH
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 │ Running GridSearchCV")
print("=" * 65)

base_params = {"random_state": RANDOM_STATE}
if imbalanced:
    base_params["class_weight"] = "balanced"

dt_base = DecisionTreeClassifier(**base_params)
cv      = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    estimator          = dt_base,
    param_grid         = param_grid,
    cv                 = cv,
    scoring            = SCORING,
    n_jobs             = -1,
    verbose            = 1,
    refit              = True,
    return_train_score = True,
)

gs.fit(X_train, y_train)

print(f"\n  Best {SCORING} (CV) : {gs.best_score_:.4f}")
print(f"  Best params :")
for k, v in gs.best_params_.items():
    print(f"    {k:<25s}: {v}")

# ══════════════════════════════════════════════════════════════════
# 4.  OVERFITTING DIAGNOSIS
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 │ Overfitting diagnosis")
print("=" * 65)

best_idx       = gs.best_index_
cv_results     = pd.DataFrame(gs.cv_results_)
train_score_cv = cv_results.loc[best_idx, "mean_train_score"]
val_score_cv   = cv_results.loc[best_idx, "mean_test_score"]

best_model     = gs.best_estimator_
y_train_prob   = best_model.predict_proba(X_train)[:, 1]
train_auc_full = roc_auc_score(y_train, y_train_prob)

gap = train_auc_full - val_score_cv
print(f"  Train AUC  (full train, refit) : {train_auc_full:.4f}")
print(f"  Train AUC  (CV mean)           : {train_score_cv:.4f}")
print(f"  Val   AUC  (CV mean)           : {val_score_cv:.4f}")
print(f"  Gap  (train_full - val_CV)     : {gap:.4f}  "
      f"→ {'Possible overfit' if gap > 0.05 else 'OK'}\n")

# ══════════════════════════════════════════════════════════════════
# 5.  TEST SET EVALUATION
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 5 │ Evaluate on Test set")
print("=" * 65)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]


metrics = {
    "accuracy" : accuracy_score(y_test, y_pred),
    "roc_auc"  : roc_auc_score(y_test, y_prob),
    "f1"       : f1_score(y_test, y_pred),
    "f1_macro" : f1_score(y_test, y_pred, average="macro"),
}

for k, v in metrics.items():
    print(f"  {k:<20s}: {v:.4f}")

print(f"\n  Classification Report:\n")
print(classification_report(y_test, y_pred))

# ══════════════════════════════════════════════════════════════════
# 6.  FEATURE IMPORTANCE
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 6 │ Feature importance (Gini impurity reduction)")
print("=" * 65)

importance_df = pd.DataFrame({
    "feature"   : feature_names,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

n_used = (importance_df["importance"] > 0).sum()
print(f"  Features used in tree : {n_used} / {len(importance_df)}")
print(f"\n  Top 20 features:")
print(importance_df.head(20).to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# 7.  PLOTS: ROC Curve + Confusion Matrix
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 7 │ Generating plots")
print("=" * 65)

plt.style.use("dark_background")
fig = plt.figure(figsize=(12, 5))
fig.patch.set_facecolor("#0d0f14")
gs_plot = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

ACCENT = "#00e5a0"
PANEL  = "#14161e"

def panel_style(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors="#c8d0e8", labelsize=8)
    ax.xaxis.label.set_color("#c8d0e8")
    ax.yaxis.label.set_color("#c8d0e8")
    ax.title.set_color("#ffffff")
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#252836")

# ── ROC Curve ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs_plot[0, 0])
RocCurveDisplay.from_predictions(
    y_test, y_prob, ax=ax1, color=ACCENT, lw=2,
    name=f"DT (AUC={metrics['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], "--", color="#5a6080", lw=1)
ax1.legend(fontsize=8, framealpha=0.2)
panel_style(ax1, "ROC Curve (Test)")

# ── Confusion Matrix ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs_plot[0, 1])
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", ax=ax2,
    cmap="YlOrRd", linewidths=0.5,
    annot_kws={"size": 13, "weight": "bold"},
    cbar_kws={"shrink": 0.8})
ax2.set_xlabel("Predicted", fontsize=9)
ax2.set_ylabel("Actual", fontsize=9)
panel_style(ax2, "Confusion Matrix (Test)")

fig.suptitle("Decision Tree — Classification Report",
             fontsize=13, fontweight="bold", color="#ffffff", y=1.02)

plot_path = f"{OUTPUT_DIR}/dt_gridsearch_report.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  → Saved plot : {plot_path}\n")

# ══════════════════════════════════════════════════════════════════
# 8.  SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 8 │ Save outputs")
print("=" * 65)

summary = {
    "best_params"       : {k: str(v) for k, v in gs.best_params_.items()},
    "cv_best_score"     : round(gs.best_score_, 6),
    "train_auc_full"    : round(train_auc_full, 6),
    "overfit_gap"       : round(gap, 6),
    "tree_depth_actual" : int(best_model.get_depth()),
    "tree_n_leaves"     : int(best_model.get_n_leaves()),
    "tree_n_nodes"      : int(best_model.tree_.node_count),
    "n_features_used"   : int(n_used),
    "test_metrics"      : {k: round(v, 6) for k, v in metrics.items()},
}
with open(f"{OUTPUT_DIR}/dt_best_params.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"  → dt_best_params.json")


print()
print("=" * 65)
print("  DT complete.")
print("=" * 65)

STEP 1 │ Load preprocessed data
  X_train : (45755, 37)
  X_test  : (19610, 37)
  y_train distribution:
icu_death_flag
0    41785
1     3970
Name: count, dtype: int64
  y_test  distribution:
icu_death_flag
0    17909
1     1701
Name: count, dtype: int64

  Positive rate (train): 0.087  → Imbalanced, will add class_weight=balanced

STEP 2 │ Define hyperparameter grid
  Total combinations : 2304
  CV folds           : 5
  Scoring metric     : roc_auc
  Total fits         : 11520

  criterion                : ['gini', 'entropy']
  max_depth                : [3, 5, 7, 10, 15, None]
  min_samples_split        : [2, 10, 20, 50]
  min_samples_leaf         : [1, 5, 10, 20]
  max_features             : [None, 'sqrt', 'log2']
  ccp_alpha                : [0.0, 0.0001, 0.001, 0.01]

STEP 3 │ Running GridSearchCV  (this may take a few minutes...)
Fitting 5 folds for each of 2304 candidates, totalling 11520 fits

  Best roc_auc (CV) : 0.8679
  Best params :
    ccp_alpha                : 0.001
    

Random Forest

In [ ]:
import numpy as np
import pandas as pd
import warnings
import json

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    f1_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
# 0.  CONFIG
# ══════════════════════════════════════════════════════════════════
DATA_DIR     = "."
OUTPUT_DIR   = "."
RANDOM_STATE = 42
CV_FOLDS     = 5
SCORING      = "roc_auc"

# ══════════════════════════════════════════════════════════════════
# 1.  LOAD DATA
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 │ Load preprocessed data")
print("=" * 65)

X_train = pd.read_csv('X_train_final.csv', index_col=0)
y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
X_test  = pd.read_csv('X_test_final.csv',  index_col=0)
y_test  = pd.read_csv('y_test_final.csv',  index_col=0)['icu_death_flag']

print(f"  X_train : {X_train.shape}")
print(f"  X_test  : {X_test.shape}")
print(f"  y_train distribution:\n{y_train.value_counts()}")
print(f"  y_test  distribution:\n{y_test.value_counts()}\n")

pos_rate   = y_train.mean()
imbalanced = pos_rate < 0.2 or pos_rate > 0.8
print(f"  Positive rate (train): {pos_rate:.3f}  "
      f"→ {'Imbalanced, will add class_weight=balanced_subsample' if imbalanced else 'Balanced'}\n")

feature_names = X_train.columns.tolist()
n_features    = len(feature_names)

# ══════════════════════════════════════════════════════════════════
# 2.  DEFINE GRID
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 2 │ Define hyperparameter grid")
print("=" * 65)

param_grid = {
    "n_estimators"      : [100, 200, 300],
    "criterion"         : ["entropy", "gini"],
    "max_depth"         : [5, 7, 10],
    "min_samples_leaf"  : [10, 20, 30],
    "min_samples_split" : [2, 10],
    "max_features"      : ["sqrt"],
    "ccp_alpha"         : [0.0, 0.001],
}

total_combos = int(np.prod([len(v) for v in param_grid.values()]))
print(f"  Total combinations : {total_combos}")
print(f"  CV folds           : {CV_FOLDS}")
print(f"  Scoring metric     : {SCORING}")
print(f"  Total fits         : {total_combos * CV_FOLDS}\n")

for k, v in param_grid.items():
    print(f"  {k:<25s}: {v}")

# ══════════════════════════════════════════════════════════════════
# 3.  GRID SEARCH
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 │ Running GridSearchCV")
print("=" * 65)

base_params = {
    "random_state" : RANDOM_STATE,
    "oob_score"    : True,
    "n_jobs"       : -1,
}
if imbalanced:
    base_params["class_weight"] = "balanced_subsample"

rf_base = RandomForestClassifier(**base_params)
cv      = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    estimator          = rf_base,
    param_grid         = param_grid,
    cv                 = cv,
    scoring            = SCORING,
    n_jobs             = -1,
    verbose            = 1,
    refit              = True,
    return_train_score = True,
)

gs.fit(X_train, y_train)
cv_results = pd.DataFrame(gs.cv_results_)

print(f"\n  Best {SCORING} (CV) : {gs.best_score_:.4f}")
print(f"  Best params :")
for k, v in gs.best_params_.items():
    print(f"    {k:<25s}: {v}")

# ══════════════════════════════════════════════════════════════════
# 4.  OOB SCORE
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 │ OOB Score  (RF-exclusive free generalization estimate)")
print("=" * 65)

best_model = gs.best_estimator_
oob_score  = best_model.oob_score_
oob_proba  = best_model.oob_decision_function_[:, 1]
oob_auc    = roc_auc_score(y_train, oob_proba)

print(f"  OOB Score (accuracy)  : {oob_score:.4f}")
print(f"  OOB AUC               : {oob_auc:.4f}")
print(f"  CV Best  ({SCORING})  : {gs.best_score_:.4f}")
print(f"\n   OOB AUC ≈ CV AUC → Generalized estimates are reliable; if discrepancies are excessive, verify the data distribution.\n")

# ══════════════════════════════════════════════════════════════════
# 5.  OVERFITTING DIAGNOSIS
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 5 │ Overfitting diagnosis")
print("=" * 65)

best_idx       = gs.best_index_
train_score_cv = cv_results.loc[best_idx, "mean_train_score"]
val_score_cv   = cv_results.loc[best_idx, "mean_test_score"]

y_train_prob   = best_model.predict_proba(X_train)[:, 1]
train_auc_full = roc_auc_score(y_train, y_train_prob)
gap            = train_auc_full - val_score_cv

print(f"  Train AUC  (full train, refit) : {train_auc_full:.4f}")
print(f"  Train AUC  (CV mean)           : {train_score_cv:.4f}")
print(f"  Val   AUC  (CV mean)           : {val_score_cv:.4f}")
print(f"  OOB   AUC  (train set)         : {oob_auc:.4f}")
print(f"  Gap   (train_full - val_CV)    : {gap:.4f}  "
      f"→ {'Possible overfit' if gap > 0.05 else 'OK'}")

# ══════════════════════════════════════════════════════════════════
# 6.  TEST SET EVALUATION
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 6 │ Evaluate on Test set")
print("=" * 65)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]


metrics = {
    "accuracy" : accuracy_score(y_test, y_pred),
    "roc_auc"  : roc_auc_score(y_test, y_prob),
    "f1"       : f1_score(y_test, y_pred),
    "f1_macro" : f1_score(y_test, y_pred, average="macro"),
}

for k, v in metrics.items():
    print(f"  {k:<20s}: {v:.4f}")

print(f"\n  Classification Report:\n")
print(classification_report(y_test, y_pred))

# ══════════════════════════════════════════════════════════════════
# 7.  FEATURE IMPORTANCE  (MDI)
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 7 │ Feature importance (MDI)")
print("=" * 65)

mdi_importance = pd.DataFrame({
    "feature"   : feature_names,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

n_used = (mdi_importance["importance"] > 0).sum()
print(f"  Features used in forest : {n_used} / {n_features}")
print(f"\n  Top 20 features:")
print(mdi_importance.head(20).to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# 8.  PLOTS: ROC Curve + Confusion Matrix
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8 │ Generating plots")
print("=" * 65)

plt.style.use("dark_background")
fig = plt.figure(figsize=(12, 5))
fig.patch.set_facecolor("#0d0f14")
gs_plot = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

ACCENT = "#00e5a0"
PANEL  = "#14161e"

def panel_style(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors="#c8d0e8", labelsize=8)
    ax.xaxis.label.set_color("#c8d0e8")
    ax.yaxis.label.set_color("#c8d0e8")
    ax.title.set_color("#ffffff")
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#252836")

# ── ROC Curve ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs_plot[0, 0])
RocCurveDisplay.from_predictions(
    y_test, y_prob, ax=ax1, color=ACCENT, lw=2,
    name=f"RF (AUC={metrics['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], "--", color="#5a6080", lw=1)
ax1.legend(fontsize=8, framealpha=0.2)
panel_style(ax1, "ROC Curve (Test)")

# ── Confusion Matrix ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs_plot[0, 1])
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", ax=ax2,
    cmap="YlOrRd", linewidths=0.5,
    annot_kws={"size": 13, "weight": "bold"},
    cbar_kws={"shrink": 0.8})
ax2.set_xlabel("Predicted", fontsize=9)
ax2.set_ylabel("Actual", fontsize=9)
panel_style(ax2, "Confusion Matrix (Test)")

fig.suptitle("Random Forest — Classification Report",
             fontsize=13, fontweight="bold", color="#ffffff", y=1.02)

plot_path = f"{OUTPUT_DIR}/rf_gridsearch_report.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  → Saved plot : {plot_path}\n")

# ══════════════════════════════════════════════════════════════════
# 9.  SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 9 │ Save outputs")
print("=" * 65)

summary = {
    "best_params"           : {k: str(v) for k, v in gs.best_params_.items()},
    "cv_best_score"         : round(gs.best_score_, 6),
    "oob_auc"               : round(oob_auc, 6),
    "train_auc_full"        : round(train_auc_full, 6),
    "overfit_gap"           : round(gap, 6),
    "test_metrics"          : {k: round(v, 6) for k, v in metrics.items()},
    "n_features_total"      : n_features,
    "n_features_nonzero_mdi": int(n_used),
}

with open(f"{OUTPUT_DIR}/rf_best_params.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"  → rf_best_params.json")


print()
print("=" * 65)
print("  RF complete.")
print("=" * 65)

STEP 1 │ Load preprocessed data
  X_train : (45755, 37)
  X_test  : (19610, 37)
  y_train distribution:
icu_death_flag
0    41785
1     3970
Name: count, dtype: int64
  y_test  distribution:
icu_death_flag
0    17909
1     1701
Name: count, dtype: int64

  Positive rate (train): 0.087  → ⚠ Imbalanced, will add class_weight=balanced_subsample

STEP 2 │ Define hyperparameter grid
  Total combinations : 432
  CV folds           : 5
  Scoring metric     : roc_auc
  Total fits         : 2160

  n_estimators             : [100, 200, 300]
  criterion                : ['entropy', 'gini']
  max_depth                : [5, 7, 10]
  min_samples_leaf         : [10, 20, 30]
  min_samples_split        : [2, 10]
  max_features             : ['sqrt', None]
  ccp_alpha                : [0.0, 0.001]

STEP 3 │ Running GridSearchCV  (this may take several minutes...)
Fitting 5 folds for each of 432 candidates, totalling 2160 fits

  ✓ Best roc_auc (CV) : 0.9096
  Best params :
    ccp_alpha                

AdaBoost

In [ ]:
import numpy as np
import pandas as pd
import warnings
import json


from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    f1_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
# 0.  CONFIG
# ══════════════════════════════════════════════════════════════════
DATA_DIR     = "."
OUTPUT_DIR   = "."
RANDOM_STATE = 42
CV_FOLDS     = 5
SCORING      = "roc_auc"

# ══════════════════════════════════════════════════════════════════
# 1.  LOAD DATA
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 │ Load preprocessed data")
print("=" * 65)

X_train = pd.read_csv('X_train_final.csv', index_col=0)
y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
X_test  = pd.read_csv('X_test_final.csv',  index_col=0)
y_test  = pd.read_csv('y_test_final.csv',  index_col=0)['icu_death_flag']

X_train = X_train.reset_index(drop=True)
y_train = y_train.reset_index(drop=True)
X_test  = X_test.reset_index(drop=True)
y_test  = y_test.reset_index(drop=True)


pos_rate   = y_train.mean()
imbalanced = pos_rate < 0.2 or pos_rate > 0.8
print(f"  Positive rate : {pos_rate:.3f}  "
      f"→ {'Imbalanced' if imbalanced else 'Balanced'}\n")

feature_names = X_train.columns.tolist()
n_features    = len(feature_names)

# ══════════════════════════════════════════════════════════════════
# 2.  DEFINE GRID
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 2 │ Define hyperparameter grid")
print("=" * 65)


def make_base_estimator(max_depth, imbalanced):
    return DecisionTreeClassifier(
        max_depth    = max_depth,
        class_weight = "balanced" if imbalanced else None,
        random_state = RANDOM_STATE,
    )


param_grid_raw = {
    "base_max_depth": [1, 2, 3],
    "n_estimators"  : [50, 100, 200, 300],
    "learning_rate" : [0.05, 0.1, 0.5],
    "algorithm"     : ["SAMME.R", "SAMME"],
}

param_grid = [
    {
        "estimator"    : [make_base_estimator(depth, imbalanced)],
        "n_estimators" : param_grid_raw["n_estimators"],
        "learning_rate": param_grid_raw["learning_rate"],
        "algorithm"    : param_grid_raw["algorithm"],
    }
    for depth in param_grid_raw["base_max_depth"]
]

total_combos = sum(
    int(np.prod([len(v) for v in g.values()]))
    for g in param_grid
)
print(f"  Total combinations : {total_combos}")
print(f"  CV folds           : {CV_FOLDS}")
print(f"  Scoring metric     : {SCORING}")
print(f"  Total fits         : {total_combos * CV_FOLDS}\n")

print(f"  base_max_depth  : {param_grid_raw['base_max_depth']}")
print(f"  n_estimators    : {param_grid_raw['n_estimators']}")
print(f"  learning_rate   : {param_grid_raw['learning_rate']}")
print(f"  algorithm       : {param_grid_raw['algorithm']}")
if imbalanced:
    print(f"Imbalanced data → Weak learners have class_weight='balanced' set")

# ══════════════════════════════════════════════════════════════════
# 3.  GRID SEARCH
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 │ Running GridSearchCV")
print("=" * 65)

ada_base = AdaBoostClassifier(random_state=RANDOM_STATE)
cv       = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True,
                           random_state=RANDOM_STATE)

gs = GridSearchCV(
    estimator          = ada_base,
    param_grid         = param_grid,
    cv                 = cv,
    scoring            = SCORING,
    n_jobs             = -1,
    verbose            = 1,
    refit              = True,
    return_train_score = True,
)


gs.fit(X_train, y_train)

cv_results = pd.DataFrame(gs.cv_results_)
best_model = gs.best_estimator_
best_depth = best_model.estimator.max_depth


print(f"  Best {SCORING} (CV) : {gs.best_score_:.4f}")
print(f"  Best params :")
print(f"    base_max_depth  : {best_depth}")
print(f"    n_estimators    : {gs.best_params_['n_estimators']}")
print(f"    learning_rate   : {gs.best_params_['learning_rate']}")
print(f"    algorithm       : {gs.best_params_['algorithm']}")

# ══════════════════════════════════════════════════════════════════
# 4.  WEAK LEARNER ANALYSIS  (AdaBoost only)
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 │ Weak Learner Analysis  (AdaBoost-exclusive)")
print("=" * 65)

estimator_weights   = best_model.estimator_weights_
estimator_errors    = best_model.estimator_errors_
n_actual_estimators = len(estimator_weights)

print(f"  Actual number of weak learners trained : {n_actual_estimators}")
print(f"  Weak learner weights (alpha): Max={estimator_weights.max():.4f} "
      f"(Round {estimator_weights.argmax()+1})  "
      f"Mean={estimator_weights.mean():.4f}")
print(f"  Weak learner errors       : Initial={estimator_errors[0]:.4f}  "
      f"final={estimator_errors[-1]:.4f}  "
      f"Mean={estimator_errors.mean():.4f}")

bad_learners = (estimator_errors >= 0.5).sum()
if bad_learners > 0:
    print(f"\n  Error rate ≥ 0.5 for {bad_learners} weak learners")
    print(f"    Recommendation: Reduce n_estimators or increase max_depth of weak learners")
else:
    print(f"\n  All weak learners have error rates < 0.5, boosting process is healthy")

# ══════════════════════════════════════════════════════════════════
# 5.  STAGED AUC
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 5 │ Staged AUC  (cumulative performance per round)")
print("=" * 65)

staged_aucs_train = [
    roc_auc_score(y_train, prob[:, 1])
    for prob in best_model.staged_predict_proba(X_train)
]
staged_aucs_test = [
    roc_auc_score(y_test, prob[:, 1])
    for prob in best_model.staged_predict_proba(X_test)
]

optimal_round = int(np.argmax(staged_aucs_test)) + 1
print(f"  Best Test AUC: Round {optimal_round} / {n_actual_estimators}")
print(f"  Train AUC : {staged_aucs_train[optimal_round-1]:.4f}")
print(f"  Test  AUC : {staged_aucs_test[optimal_round-1]:.4f}")

# ══════════════════════════════════════════════════════════════════
# 6.  OVERFITTING DIAGNOSIS
# ════════════════════════════════════
print("STEP 6 │ Overfitting diagnosis")
print("=" * 65)

best_idx       = gs.best_index_
train_score_cv = cv_results.loc[best_idx, "mean_train_score"]
val_score_cv   = cv_results.loc[best_idx, "mean_test_score"]

y_train_prob   = best_model.predict_proba(X_train)[:, 1]
train_auc_full = roc_auc_score(y_train, y_train_prob)
gap            = train_auc_full - val_score_cv

print(f"  Train AUC (full train refit) : {train_auc_full:.4f}")
print(f"  Train AUC (CV mean)          : {train_score_cv:.4f}")
print(f"  Val   AUC (CV mean)          : {val_score_cv:.4f}")
print(f"  Gap  (train_full - val_CV)   : {gap:.4f}  "
      f"→ {'Possible overfit' if gap > 0.05 else 'OK'}")

print(f"    When overfitting occurs, consider: Reducing n_estimators or learning_rate\n")

# ══════════════════════════════════════════════════════════════════
# 7.  TEST SET EVALUATION
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 7 │ Evaluate on Test set")
print("=" * 65)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]


metrics = {
    "accuracy" : accuracy_score(y_test, y_pred),
    "roc_auc"  : roc_auc_score(y_test, y_prob),
    "f1"       : f1_score(y_test, y_pred),
    "f1_macro" : f1_score(y_test, y_pred, average="macro"),
}

for k, v in metrics.items():
    print(f"  {k:<20s}: {v:.4f}")

print(f"\n  Classification Report:\n")
print(classification_report(y_test, y_pred))

# ══════════════════════════════════════════════════════════════════
# 8.  FEATURE IMPORTANCE  (MDI)
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 8 │ Feature importance (MDI — weighted avg over weak learners)")
print("=" * 65)

mdi_df = pd.DataFrame({
    "feature"   : feature_names,
    "importance": best_model.feature_importances_,
}).sort_values("importance", ascending=False).reset_index(drop=True)

n_used = (mdi_df["importance"] > 0).sum()
print(f"  Used features: {n_used} / {n_features}")
print(f"\n  Top 20 features:")
print(mdi_df.head(20).to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# 9.  PLOTS: ROC Curve + Confusion Matrix
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 9 │ Generating plots")
print("=" * 65)

plt.style.use("dark_background")
fig = plt.figure(figsize=(12, 5))
fig.patch.set_facecolor("#0d0f14")
gs_plot = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

ACCENT = "#00e5a0"
PANEL  = "#14161e"

def panel_style(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors="#c8d0e8", labelsize=8)
    ax.xaxis.label.set_color("#c8d0e8")
    ax.yaxis.label.set_color("#c8d0e8")
    ax.title.set_color("#ffffff")
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#252836")

# ── ROC Curve ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs_plot[0, 0])
RocCurveDisplay.from_predictions(
    y_test, y_prob, ax=ax1, color=ACCENT, lw=2,
    name=f"AdaBoost (AUC={metrics['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], "--", color="#5a6080", lw=1)
ax1.legend(fontsize=8, framealpha=0.2)
panel_style(ax1, "ROC Curve (Test)")

# ── Confusion Matrix ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs_plot[0, 1])
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", ax=ax2,
    cmap="YlOrRd", linewidths=0.5,
    annot_kws={"size": 13, "weight": "bold"},
    cbar_kws={"shrink": 0.8})
ax2.set_xlabel("Predicted", fontsize=9)
ax2.set_ylabel("Actual", fontsize=9)
panel_style(ax2, "Confusion Matrix (Test)")

fig.suptitle("AdaBoost — Classification Report",
             fontsize=13, fontweight="bold", color="#ffffff", y=1.02)

plot_path = f"{OUTPUT_DIR}/adaboost_gridsearch_report.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  → Saved plot : {plot_path}\n")

# ══════════════════════════════════════════════════════════════════
# 10.  SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 10 │ Save outputs")
print("=" * 65)

summary = {
    "best_params": {
        "base_max_depth": int(best_depth),
        "n_estimators"  : int(gs.best_params_["n_estimators"]),
        "learning_rate" : float(gs.best_params_["learning_rate"]),
        "algorithm"     : gs.best_params_["algorithm"],
    },
    "cv_best_score"         : round(gs.best_score_, 6),
    "train_auc_full"        : round(train_auc_full, 6),
    "overfit_gap"           : round(gap, 6),
    "optimal_round"         : int(optimal_round),
    "staged_auc_at_optimal" : round(staged_aucs_test[optimal_round - 1], 6),
    "n_actual_estimators"   : int(n_actual_estimators),
    "n_failed_learners"     : int(bad_learners),
    "test_metrics"          : {k: round(v, 6) for k, v in metrics.items()},
    "n_features_total"      : n_features,
    "n_features_mdi_nonzero": int(n_used)
}

with open(f"{OUTPUT_DIR}/adaboost_best_params.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"  → adaboost_best_params.json")



print()
print("=" * 65)
print("  AdaBoost Grid Search completed.")
print("=" * 65)

STEP 1 │ Load preprocessed data
  X_train : (45755, 37)
  X_test  : (19610, 37)
  y_train distribution:
icu_death_flag
0    41785
1     3970
Name: count, dtype: int64
  y_test  distribution:
icu_death_flag
0    17909
1     1701
Name: count, dtype: int64

  Positive rate : 0.087  → Imbalanced

STEP 2 │ Define hyperparameter grid
  Total combinations : 72
  CV folds           : 5
  Scoring metric     : roc_auc
  Total fits         : 360

  base_max_depth  : [1, 2, 3]
  n_estimators    : [50, 100, 200, 300]
  learning_rate   : [0.05, 0.1, 0.5]
  algorithm       : ['SAMME.R', 'SAMME']
Imbalanced data → Weak learners have class_weight='balanced' set

STEP 3 │ Running GridSearchCV
Fitting 5 folds for each of 72 candidates, totalling 360 fits
  Best roc_auc (CV) : 0.9162
  Best params :
    base_max_depth  : 2
    n_estimators    : 200
    learning_rate   : 0.1
    algorithm       : SAMME.R

STEP 4 │ Weak Learner Analysis  (AdaBoost-exclusive)
  Actual number of weak learners trained : 200
  

XGBoost

In [ ]:
import numpy as np
import pandas as pd
import warnings
import json

import xgboost as xgb
from xgboost import XGBClassifier

from sklearn.model_selection import GridSearchCV, StratifiedKFold, train_test_split as tts
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    f1_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
# 0.  CONFIG
# ══════════════════════════════════════════════════════════════════
DATA_DIR     = "."
OUTPUT_DIR   = "."
RANDOM_STATE = 42
CV_FOLDS     = 5
SCORING      = "roc_auc"

# ══════════════════════════════════════════════════════════════════
# 1.  LOAD DATA
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 │ Load preprocessed data")
print("=" * 65)

X_train = pd.read_csv('X_train_final.csv', index_col=0)
y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
X_test  = pd.read_csv('X_test_final.csv',  index_col=0)
y_test  = pd.read_csv('y_test_final.csv',  index_col=0)['icu_death_flag']


pos_rate   = y_train.mean()
neg_rate   = 1 - pos_rate
imbalanced = pos_rate < 0.2 or pos_rate > 0.8

scale_pos_weight = neg_rate / pos_rate if imbalanced else 1.0

print(f"  Positive rate        : {pos_rate:.3f}")
print(f"  scale_pos_weight     : {scale_pos_weight:.2f}  "
      f"{'Imbalanced, applied' if imbalanced else 'Balanced, set to 1.0'}\n")

feature_names = X_train.columns.tolist()
n_features    = len(feature_names)

# ══════════════════════════════════════════════════════════════════
# 2.  DEFINE GRID
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 2 │ Define hyperparameter grid")
print("=" * 65)


param_grid = {
    "n_estimators"    : [100, 200],
    "max_depth"       : [3, 4, 5, 6],
    "learning_rate"   : [0.05, 0.1],
    "subsample"       : [0.8],
    "colsample_bytree": [0.8],
    "min_child_weight": [1, 5, 10],
    "gamma"           : [0, 0.1],
    "reg_alpha"       : [0],
    "reg_lambda"      : [1.0],
}

total_combos = int(np.prod([len(v) for v in param_grid.values()]))
print(f"  Total combinations : {total_combos:,}")
print(f"  CV folds           : {CV_FOLDS}")
print(f"  Scoring metric     : {SCORING}")
print(f"  Total fits         : {total_combos * CV_FOLDS:,}\n")

for k, v in param_grid.items():
    print(f"  {k:<25s}: {v}")

# ══════════════════════════════════════════════════════════════════
# 3.  GRID SEARCH
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 │ Running GridSearchCV")
print("=" * 65)

base_params = {
    "objective"        : "binary:logistic",
    "eval_metric"      : "auc",
    "scale_pos_weight" : scale_pos_weight,
    "random_state"     : RANDOM_STATE,
    "n_jobs"           : -1,
    "verbosity"        : 0,
    "use_label_encoder": False,
}

xgb_base = XGBClassifier(**base_params)
cv       = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    estimator          = xgb_base,
    param_grid         = param_grid,
    cv                 = cv,
    scoring            = SCORING,
    n_jobs             = -1,
    verbose            = 1,
    refit              = True,
    return_train_score = True,
)

gs.fit(X_train, y_train)
cv_results = pd.DataFrame(gs.cv_results_)

print(f"\n  Best {SCORING} (CV) : {gs.best_score_:.4f}")
print(f"  Best params :")
for k, v in gs.best_params_.items():
    print(f"    {k:<25s}: {v}")

# ══════════════════════════════════════════════════════════════════
# 4.  EARLY STOPPING VALIDATION  (find optimal n_estimators)
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 │ Early Stopping Validation  (find optimal n_estimators)")
print("=" * 65)

X_tr_es, X_val_es, y_tr_es, y_val_es = tts(
    X_train, y_train,
    test_size=0.2, stratify=y_train, random_state=RANDOM_STATE
)

best_p_es = {k: v for k, v in gs.best_params_.items() if k != "n_estimators"}
xgb_es = XGBClassifier(
    **best_p_es,
    **base_params,
    n_estimators          = 1000,
    early_stopping_rounds = 30,
)

xgb_es.fit(
    X_tr_es, y_tr_es,
    eval_set = [(X_tr_es, y_tr_es), (X_val_es, y_val_es)],
    verbose  = 50
)

optimal_n_estimators = xgb_es.best_iteration + 1
es_results           = xgb_es.evals_result()
train_auc_curve      = es_results["validation_0"]["auc"]
val_auc_curve        = es_results["validation_1"]["auc"]

print(f"\n  Early stopping → optimal n_estimators : {optimal_n_estimators}")
print(f"  Best val AUC at optimal round         : {xgb_es.best_score:.4f}")

# Retrain on the full training set using the optimal n_estimators. 
print(f"\n  Retraining final model with n_estimators={optimal_n_estimators}...")
final_params = {**gs.best_params_, "n_estimators": optimal_n_estimators}
best_model   = XGBClassifier(**final_params, **base_params)
best_model.fit(X_train, y_train)
print(f"  ✓ Final model trained.\n")

# ══════════════════════════════════════════════════════════════════
# 5.  OVERFITTING DIAGNOSIS
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 5 │ Overfitting diagnosis")
print("=" * 65)

best_idx       = gs.best_index_
train_score_cv = cv_results.loc[best_idx, "mean_train_score"]
val_score_cv   = cv_results.loc[best_idx, "mean_test_score"]

y_train_prob   = best_model.predict_proba(X_train)[:, 1]
train_auc_full = roc_auc_score(y_train, y_train_prob)
gap            = train_auc_full - val_score_cv

print(f"  Train AUC  (full train, final model) : {train_auc_full:.4f}")
print(f"  Train AUC  (CV mean)                 : {train_score_cv:.4f}")
print(f"  Val   AUC  (CV mean)                 : {val_score_cv:.4f}")
print(f"  Gap   (train_full - val_CV)          : {gap:.4f}  "
      f"→ {'Possible overfit' if gap > 0.05 else 'OK'}\n")

# ══════════════════════════════════════════════════════════════════
# 6.  TEST SET EVALUATION
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 6 │ Evaluate on Test set")
print("=" * 65)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

# 只保留核心分类评估指标
metrics = {
    "accuracy" : accuracy_score(y_test, y_pred),
    "roc_auc"  : roc_auc_score(y_test, y_prob),
    "f1"       : f1_score(y_test, y_pred),
    "f1_macro" : f1_score(y_test, y_pred, average="macro"),
}

for k, v in metrics.items():
    print(f"  {k:<20s}: {v:.4f}")

print(f"\n  Classification Report:\n")
print(classification_report(y_test, y_pred))

# ══════════════════════════════════════════════════════════════════
# 7.  FEATURE IMPORTANCE  (Gain)
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 7 │ Feature importance (Gain)")
print("=" * 65)

imp_gain     = best_model.get_booster().get_score(importance_type="gain")
imp_gain_df  = pd.DataFrame(
    [(f, imp_gain.get(f, 0)) for f in feature_names],
    columns=["feature", "imp_gain"]
).sort_values("imp_gain", ascending=False).reset_index(drop=True)

n_used = (imp_gain_df["imp_gain"] > 0).sum()
print(f"  used features : {n_used} / {n_features}")
print(f"\n  Top 20 features:")
print(imp_gain_df.head(20).to_string(index=False))

# ══════════════════════════════════════════════════════════════════
# 8.  PLOTS  ── ROC Curve + Confusion Matrix
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 8 │ Generating plots")
print("=" * 65)

plt.style.use("dark_background")
fig = plt.figure(figsize=(12, 5))
fig.patch.set_facecolor("#0d0f14")
gs_plot = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

ACCENT = "#00e5a0"
PANEL  = "#14161e"

def panel_style(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors="#c8d0e8", labelsize=8)
    ax.xaxis.label.set_color("#c8d0e8")
    ax.yaxis.label.set_color("#c8d0e8")
    ax.title.set_color("#ffffff")
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#252836")

# ── ROC Curve ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs_plot[0, 0])
RocCurveDisplay.from_predictions(
    y_test, y_prob, ax=ax1, color=ACCENT, lw=2,
    name=f"XGBoost (AUC={metrics['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], "--", color="#5a6080", lw=1)
ax1.legend(fontsize=8, framealpha=0.2)
panel_style(ax1, "ROC Curve (Test)")

# ── Confusion Matrix ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs_plot[0, 1])
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", ax=ax2,
    cmap="YlOrRd", linewidths=0.5,
    annot_kws={"size": 13, "weight": "bold"},
    cbar_kws={"shrink": 0.8})
ax2.set_xlabel("Predicted", fontsize=9)
ax2.set_ylabel("Actual", fontsize=9)
panel_style(ax2, "Confusion Matrix (Test)")

fig.suptitle("XGBoost — Classification Report",
             fontsize=13, fontweight="bold", color="#ffffff", y=1.02)

plot_path = f"{OUTPUT_DIR}/xgb_gridsearch_report.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  → Saved plot : {plot_path}\n")

# ══════════════════════════════════════════════════════════════════
# 9.  SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 9 │ Save outputs")
print("=" * 65)

summary = {
    "best_params_grid"      : {k: str(v) for k, v in gs.best_params_.items()},
    "optimal_n_estimators"  : optimal_n_estimators,
    "final_params"          : {k: str(v) for k, v in final_params.items()},
    "cv_best_score"         : round(gs.best_score_, 6),
    "early_stop_val_auc"    : round(float(xgb_es.best_score), 6),
    "train_auc_full"        : round(train_auc_full, 6),
    "overfit_gap"           : round(gap, 6),
    "test_metrics"          : {k: round(v, 6) for k, v in metrics.items()},
    "n_features_total"      : n_features,
    "n_features_gain_nonzero": int(n_used),
}

with open(f"{OUTPUT_DIR}/xgb_best_params.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"  → xgb_best_params.json")


print()
print("=" * 65)
print("  XGBoost Grid Search complete.")
print(f"  CV AUC             : {gs.best_score_:.4f}")
print(f"  Early-stop Val AUC : {xgb_es.best_score:.4f}  @ round {optimal_n_estimators}")
print(f"  Test AUC           : {metrics['roc_auc']:.4f}")
print(f"  Overfit gap        : {gap:.4f}")
print(f"  Selected features  : {len(selected_features)}")
print("=" * 65)

STEP 1 │ Load preprocessed data
  X_train : (45755, 37)
  X_test  : (19610, 37)
  y_train distribution:
icu_death_flag
0    41785
1     3970
Name: count, dtype: int64
  y_test  distribution:
icu_death_flag
0    17909
1     1701
Name: count, dtype: int64

  Positive rate        : 0.087
  scale_pos_weight     : 10.53  ⚠ Imbalanced, applied

STEP 2 │ Define hyperparameter grid
  Total combinations : 96
  CV folds           : 5
  Scoring metric     : roc_auc
  Total fits         : 480

  n_estimators             : [100, 200]
  max_depth                : [3, 4, 5, 6]
  learning_rate            : [0.05, 0.1]
  subsample                : [0.8]
  colsample_bytree         : [0.8]
  min_child_weight         : [1, 5, 10]
  gamma                    : [0, 0.1]
  reg_alpha                : [0]
  reg_lambda               : [1.0]

STEP 3 │ Running GridSearchCV
Fitting 5 folds for each of 96 candidates, totalling 480 fits

  ✓ Best roc_auc (CV) : 0.9197
  Best params :
    colsample_bytree         : 0.

SVM

In [ ]:
import numpy as np
import pandas as pd
import warnings
import json

from sklearn.svm import SVC
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    f1_score, classification_report, confusion_matrix,
    RocCurveDisplay
)
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")

# ══════════════════════════════════════════════════════════════════
# 0.  CONFIG
# ══════════════════════════════════════════════════════════════════
DATA_DIR     = "."
OUTPUT_DIR   = "."
RANDOM_STATE = 42
CV_FOLDS     = 5
SCORING      = "roc_auc"

# ══════════════════════════════════════════════════════════════════
# 1.  LOAD DATA
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 │ Load preprocessed data")
print("=" * 65)

X_train = pd.read_csv('X_train_final.csv', index_col=0)
y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
X_test  = pd.read_csv('X_test_final.csv',  index_col=0)
y_test  = pd.read_csv('y_test_final.csv',  index_col=0)['icu_death_flag']


pos_rate   = y_train.mean()
imbalanced = pos_rate < 0.2 or pos_rate > 0.8
print(f"  Positive rate (train): {pos_rate:.3f}  "
      f"→ {'Imbalanced, using class_weight=balanced' if imbalanced else 'Balanced'}\n")

# ══════════════════════════════════════════════════════════════════
# 2.  DEFINE GRID
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 2 │ Define hyperparameter grid")
print("=" * 65)


param_grid = {
    "C"      : [0.1, 1],
    "kernel" : ["rbf"],
    "gamma"  : ["auto"],
}

total_combos = int(np.prod([len(v) for v in param_grid.values()]))
print(f"  Total combinations : {total_combos}")
print(f"  CV folds           : {CV_FOLDS}")
print(f"  Scoring metric     : {SCORING}")
print(f"  Total fits         : {total_combos * CV_FOLDS}\n")

for k, v in param_grid.items():
    print(f"  {k:<10s}: {v}")

# ══════════════════════════════════════════════════════════════════
# 3.  GRID SEARCH
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 3 │ Running GridSearchCV")
print("=" * 65)

svm_base = SVC(
    class_weight = "balanced",
    probability  = True,        
    random_state = RANDOM_STATE,
)

cv = StratifiedKFold(n_splits=CV_FOLDS, shuffle=True, random_state=RANDOM_STATE)

gs = GridSearchCV(
    estimator          = svm_base,
    param_grid         = param_grid,
    cv                 = cv,
    scoring            = SCORING,
    n_jobs             = -1,
    verbose            = 1,
    refit              = True,
    return_train_score = True,
)

gs.fit(X_train, y_train)
cv_results = pd.DataFrame(gs.cv_results_)

print(f"\n  Best {SCORING} (CV) : {gs.best_score_:.4f}")
print(f"  Best params :")
for k, v in gs.best_params_.items():
    print(f"    {k:<10s}: {v}")

# ══════════════════════════════════════════════════════════════════
# 4.  OVERFITTING DIAGNOSIS
# ══════════════════════════════════════════════════════════════════
print("\n" + "=" * 65)
print("STEP 4 │ Overfitting diagnosis")
print("=" * 65)

best_idx       = gs.best_index_
train_score_cv = cv_results.loc[best_idx, "mean_train_score"]
val_score_cv   = cv_results.loc[best_idx, "mean_test_score"]

best_model     = gs.best_estimator_
y_train_prob   = best_model.predict_proba(X_train)[:, 1]
train_auc_full = roc_auc_score(y_train, y_train_prob)
gap            = train_auc_full - val_score_cv

print(f"  Train AUC  (full train, refit) : {train_auc_full:.4f}")
print(f"  Train AUC  (CV mean)           : {train_score_cv:.4f}")
print(f"  Val   AUC  (CV mean)           : {val_score_cv:.4f}")
print(f"  Gap  (train_full - val_CV)     : {gap:.4f}  "
      f"→ {'Possible overfit' if gap > 0.05 else 'OK'}\n")

# ══════════════════════════════════════════════════════════════════
# 5.  TEST SET EVALUATION
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 5 │ Evaluate on Test set")
print("=" * 65)

y_pred = best_model.predict(X_test)
y_prob = best_model.predict_proba(X_test)[:, 1]

metrics = {
    "accuracy" : accuracy_score(y_test, y_pred),
    "roc_auc"  : roc_auc_score(y_test, y_prob),
    "f1"       : f1_score(y_test, y_pred),
    "f1_macro" : f1_score(y_test, y_pred, average="macro"),
}

for k, v in metrics.items():
    print(f"  {k:<20s}: {v:.4f}")

print(f"\n  Classification Report:\n")
print(classification_report(y_test, y_pred))

# ══════════════════════════════════════════════════════════════════
# 6.  PLOTS  ── ROC Curve + Confusion Matrix
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 6 │ Generating plots")
print("=" * 65)

plt.style.use("dark_background")
fig = plt.figure(figsize=(12, 5))
fig.patch.set_facecolor("#0d0f14")
gs_plot = gridspec.GridSpec(1, 2, figure=fig, wspace=0.35)

ACCENT = "#00e5a0"
PANEL  = "#14161e"

def panel_style(ax, title):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors="#c8d0e8", labelsize=8)
    ax.xaxis.label.set_color("#c8d0e8")
    ax.yaxis.label.set_color("#c8d0e8")
    ax.title.set_color("#ffffff")
    ax.set_title(title, fontsize=10, fontweight="bold", pad=8)
    for spine in ax.spines.values():
        spine.set_edgecolor("#252836")

# ── ROC Curve ─────────────────────────────────────────────────────
ax1 = fig.add_subplot(gs_plot[0, 0])
RocCurveDisplay.from_predictions(
    y_test, y_prob, ax=ax1, color=ACCENT, lw=2,
    name=f"SVM (AUC={metrics['roc_auc']:.3f})")
ax1.plot([0, 1], [0, 1], "--", color="#5a6080", lw=1)
ax1.legend(fontsize=8, framealpha=0.2)
panel_style(ax1, "ROC Curve (Test)")

# ── Confusion Matrix ──────────────────────────────────────────────
ax2 = fig.add_subplot(gs_plot[0, 1])
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", ax=ax2,
    cmap="YlOrRd", linewidths=0.5,
    annot_kws={"size": 13, "weight": "bold"},
    cbar_kws={"shrink": 0.8})
ax2.set_xlabel("Predicted", fontsize=9)
ax2.set_ylabel("Actual", fontsize=9)
panel_style(ax2, "Confusion Matrix (Test)")

fig.suptitle("SVM — Classification Report",
             fontsize=13, fontweight="bold", color="#ffffff", y=1.02)

plot_path = f"{OUTPUT_DIR}/svm_gridsearch_report.png"
plt.savefig(plot_path, dpi=150, bbox_inches="tight",
            facecolor=fig.get_facecolor())
plt.close()
print(f"  → Saved plot : {plot_path}\n")

# ══════════════════════════════════════════════════════════════════
# 7.  SAVE OUTPUTS
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 7 │ Save outputs")
print("=" * 65)

summary = {
    "best_params"   : gs.best_params_,
    "cv_best_score" : round(gs.best_score_, 6),
    "train_auc_full": round(train_auc_full, 6),
    "overfit_gap"   : round(gap, 6),
    "test_metrics"  : {k: round(v, 6) for k, v in metrics.items()},
}

with open(f"{OUTPUT_DIR}/svm_best_params.json", "w") as f:
    json.dump(summary, f, indent=2)
print(f"  → svm_best_params.json")


print("=" * 65)
print("  Grid Search complete.")
print(f"  Best model : {gs.best_params_}")
print(f"  Test AUC   : {metrics['roc_auc']:.4f}")
print(f"  Overfit gap: {gap:.4f}")
print("=" * 65)

STEP 1 │ Load preprocessed data
  X_train : (45755, 37)
  X_test  : (19610, 37)
  y_train distribution:
icu_death_flag
0    41785
1     3970
Name: count, dtype: int64
  y_test  distribution:
icu_death_flag
0    17909
1     1701
Name: count, dtype: int64

  Positive rate (train): 0.087  → ⚠ Imbalanced, using class_weight=balanced

STEP 2 │ Define hyperparameter grid
  Total combinations : 2
  CV folds           : 5
  Scoring metric     : roc_auc
  Total fits         : 10

  C         : [0.1, 1]
  kernel    : ['rbf']
  gamma     : ['auto']

STEP 3 │ Running GridSearchCV
Fitting 5 folds for each of 2 candidates, totalling 10 fits

  ✓ Best roc_auc (CV) : 0.9130
  Best params :
    C         : 0.1
    gamma     : auto
    kernel    : rbf

STEP 4 │ Overfitting diagnosis
  Train AUC  (full train, refit) : 0.9253
  Train AUC  (CV mean)           : 0.9255
  Val   AUC  (CV mean)           : 0.9130
  Gap  (train_full - val_CV)     : 0.0123  → ✓ OK

STEP 5 │ Evaluate on Test set
  accuracy       

### Comparison

In [ ]:
import numpy as np
import pandas as pd
import json
import warnings
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec

from sklearn.metrics import (
    accuracy_score, roc_auc_score,
    precision_score, recall_score,
    f1_score, classification_report
)

warnings.filterwarnings("ignore")

OUTPUT_DIR = "."

# ══════════════════════════════════════════════════════════════════
# 1.  LOAD TEST PREDICTIONS AND EVALUATE
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 1 │ Load data & rebuild predictions for each model")
print("=" * 65)


y_test = pd.read_csv('y_test_final.csv', index_col=0)['icu_death_flag']


def _is_imbalanced(y_train):
    pos_rate = y_train.mean()
    return pos_rate < 0.2 or pos_rate > 0.8

def load_best_model_lr():
    """Logistic Regression"""
    from sklearn.linear_model import LogisticRegression
    X_train = pd.read_csv('X_train_final.csv', index_col=0)
    y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
    X_test  = pd.read_csv('X_test_final.csv',  index_col=0)

    with open('lr_best_params.json') as f:
        info = json.load(f)

    p = info["best_params"]

    cw = "balanced" if _is_imbalanced(y_train) else None
    model = LogisticRegression(
        penalty      = p["penalty"],
        C            = float(p["C"]),
        solver       = p["solver"],
        max_iter     = 2000,
        random_state = 42,
        class_weight = cw,
        **({"l1_ratio": float(p["l1_ratio"])} if "l1_ratio" in p else {})
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return y_pred, y_prob

def load_best_model_dt():
    """Decision Tree"""
    from sklearn.tree import DecisionTreeClassifier
    X_train = pd.read_csv('X_train_final.csv', index_col=0)
    y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
    X_test  = pd.read_csv('X_test_final.csv',  index_col=0)

    with open('dt_best_params.json') as f:
        info = json.load(f)

    p  = info["best_params"]
    cw = "balanced" if _is_imbalanced(y_train) else None
    model = DecisionTreeClassifier(
        criterion         = p["criterion"],
        max_depth         = None if p["max_depth"] == "None" else int(p["max_depth"]),
        min_samples_split = int(p["min_samples_split"]),
        min_samples_leaf  = int(p["min_samples_leaf"]),
        max_features      = None if p["max_features"] == "None" else p["max_features"],
        ccp_alpha         = float(p["ccp_alpha"]),
        random_state      = 42,
        class_weight      = cw,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return y_pred, y_prob

def load_best_model_rf():
    """Random Forest — class_weight=balanced_subsample"""
    from sklearn.ensemble import RandomForestClassifier
    X_train = pd.read_csv('X_train_final.csv', index_col=0)
    y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
    X_test  = pd.read_csv('X_test_final.csv',  index_col=0)

    with open('rf_best_params.json') as f:
        info = json.load(f)

    p  = info["best_params"]
    # The RF modeling script uses balanced_subsample (not balanced).
    cw = "balanced_subsample" if _is_imbalanced(y_train) else None
    model = RandomForestClassifier(
        n_estimators      = int(p["n_estimators"]),
        criterion         = p["criterion"],
        max_depth         = None if p["max_depth"] == "None" else int(p["max_depth"]),
        min_samples_leaf  = int(p["min_samples_leaf"]),
        min_samples_split = int(p["min_samples_split"]),
        max_features      = None if p["max_features"] == "None" else p["max_features"],
        ccp_alpha         = float(p["ccp_alpha"]),
        random_state      = 42,
        n_jobs            = -1,
        oob_score         = True,
        class_weight      = cw,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return y_pred, y_prob

def load_best_model_ada():
    """AdaBoost"""
    from sklearn.ensemble import AdaBoostClassifier
    from sklearn.tree import DecisionTreeClassifier
    X_train = pd.read_csv('X_train_final.csv', index_col=0)
    y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
    X_test  = pd.read_csv('X_test_final.csv',  index_col=0)

    with open('adaboost_best_params.json') as f:
        info = json.load(f)

    p         = info["best_params"]
    pos_rate  = y_train.mean()
    imbalanced = pos_rate < 0.2 or pos_rate > 0.8

    base = DecisionTreeClassifier(
        max_depth    = int(p["base_max_depth"]),
        class_weight = "balanced" if imbalanced else None,
        random_state = 42,
    )
    model = AdaBoostClassifier(
        estimator    = base,
        n_estimators = int(p["n_estimators"]),
        learning_rate= float(p["learning_rate"]),
        algorithm    = p["algorithm"],
        random_state = 42,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return y_pred, y_prob

def load_best_model_xgb():
    """XGBoost"""
    from xgboost import XGBClassifier
    X_train = pd.read_csv('X_train_final.csv', index_col=0)
    y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
    X_test  = pd.read_csv('X_test_final.csv',  index_col=0)

    with open('xgb_best_params.json') as f:
        info = json.load(f)

    p        = info["final_params"]  
    pos_rate = y_train.mean()
    spw      = (1 - pos_rate) / pos_rate if (pos_rate < 0.2 or pos_rate > 0.8) else 1.0

    model = XGBClassifier(
        n_estimators      = int(p["n_estimators"]),
        max_depth         = int(p["max_depth"]),
        learning_rate     = float(p["learning_rate"]),
        subsample         = float(p["subsample"]),
        colsample_bytree  = float(p["colsample_bytree"]),
        min_child_weight  = int(p["min_child_weight"]),
        gamma             = float(p["gamma"]),
        reg_alpha         = float(p["reg_alpha"]),
        reg_lambda        = float(p["reg_lambda"]),
        objective         = "binary:logistic",
        eval_metric       = "auc",
        scale_pos_weight  = spw,
        random_state      = 42,
        n_jobs            = -1,
        verbosity         = 0,
        use_label_encoder = False,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return y_pred, y_prob

def load_best_model_svm():
    """SVM"""
    from sklearn.svm import SVC
    X_train = pd.read_csv('X_train_final.csv', index_col=0)
    y_train = pd.read_csv('y_train_final.csv', index_col=0)['icu_death_flag']
    X_test  = pd.read_csv('X_test_final.csv',  index_col=0)

    with open('svm_best_params.json') as f:
        info = json.load(f)

    p = info["best_params"]
    model = SVC(
        C            = float(p["C"]),
        kernel       = p["kernel"],
        gamma        = p["gamma"],
        class_weight = "balanced",
        probability  = True,
        random_state = 42,
    )
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    return y_pred, y_prob

# ══════════════════════════════════════════════════════════════════
# 2.  COMPUTE METRICS FOR ALL MODELS
# ══════════════════════════════════════════════════════════════════
print("=" * 65)
print("STEP 2 │ Computing metrics for all models")
print("=" * 65)

MODEL_LOADERS = {
    "Logistic Regression" : load_best_model_lr,
    "Decision Tree"       : load_best_model_dt,
    "Random Forest"       : load_best_model_rf,
    "AdaBoost"            : load_best_model_ada,
    "XGBoost"             : load_best_model_xgb,
    "SVM"                 : load_best_model_svm,
}

METRICS_COLS = ["Accuracy", "ROC-AUC", "Precision", "Recall", "F1-Score", "Macro F1"]

rows = []

for model_name, loader in MODEL_LOADERS.items():
    print(f"  Processing: {model_name} ...", end=" ", flush=True)
    try:
        y_pred, y_prob = loader()
        row = {
            "Model"     : model_name,
            "Accuracy"  : accuracy_score(y_test, y_pred),
            "ROC-AUC"   : roc_auc_score(y_test, y_prob),
            "Precision" : precision_score(y_test, y_pred, zero_division=0),
            "Recall"    : recall_score(y_test, y_pred, zero_division=0),
            "F1-Score"  : f1_score(y_test, y_pred, zero_division=0),
            "Macro F1"  : f1_score(y_test, y_pred, average="macro", zero_division=0),
        }
        rows.append(row)
        print(f"  AUC={row['ROC-AUC']:.4f}  F1={row['F1-Score']:.4f}")
    except Exception as e:
        print(f"  ERROR: {e}")

results_df = pd.DataFrame(rows).set_index("Model")
results_df = results_df[METRICS_COLS]

print()
print("=" * 65)
print("  Full comparison table:")
print("=" * 65)
print(results_df.round(4).to_string())

# ══════════════════════════════════════════════════════════════════
# 3.  HIGHLIGHT BEST PER METRIC
# ══════════════════════════════════════════════════════════════════
print()
print("=" * 65)
print("  Best model per metric:")
print("=" * 65)
for col in METRICS_COLS:
    best_model = results_df[col].idxmax()
    best_val   = results_df[col].max()
    print(f"  {col:<15s}: {best_model:<25s} ({best_val:.4f})")

# ══════════════════════════════════════════════════════════════════
# 4.  SAVE CSV
# ══════════════════════════════════════════════════════════════════
csv_path = f"{OUTPUT_DIR}/model_comparison.csv"
results_df.round(4).to_csv(csv_path)
print(f"\n  → Saved : {csv_path}")

print("=" * 65)
print("  Comparison complete.")
print("=" * 65)

STEP 1 │ Load data & rebuild predictions for each model
STEP 2 │ Computing metrics for all models
  Processing: Logistic Regression ... ✓  AUC=0.9058  F1=0.4663
  Processing: Decision Tree ... ✓  AUC=0.8669  F1=0.3649
  Processing: Random Forest ... ✓  AUC=0.9119  F1=0.5001
  Processing: AdaBoost ... ✓  AUC=0.9206  F1=0.4965
  Processing: XGBoost ... ✓  AUC=0.9235  F1=0.5034
  Processing: SVM ... ✓  AUC=0.9157  F1=0.4702

  Full comparison table:
                     Accuracy  ROC-AUC  Precision  Recall  F1-Score  Macro F1
Model                                                                        
Logistic Regression    0.8380   0.9058     0.3264  0.8160    0.4663    0.6854
Decision Tree          0.7482   0.8669     0.2335  0.8336    0.3649    0.6039
Random Forest          0.8680   0.9119     0.3723  0.7613    0.5001    0.7120
AdaBoost               0.8569   0.9206     0.3573  0.8136    0.4965    0.7065
XGBoost                0.8603   0.9235     0.3639  0.8160    0.5034    0.7110
SVM